## MCP Servers with Foundry Agents

![lab_flow](./Assets/lab_flow.png)

### Installing Required Libraries

In [1]:
%pip install azure-ai-projects==2.0.0b2 openai==1.109.1 python-dotenv azure-identity

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\zvijayfa\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


### Setting Up the Environment Variables

In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, MCPTool, Tool

load_dotenv()

foundry_project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")
mcp_server_name = os.getenv("MCP_SERVER_NAME")

### Setting up the Foundry Project Client

In [3]:
project_client = AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=DefaultAzureCredential(),
)

### Creating the OpenAI Client

In [4]:
openai_client = project_client.get_openai_client()

### Getting the MCP Server Connection ID

In [5]:
connection_id = ""

for connection in project_client.connections.list():
    if connection.name == mcp_server_name:
        connection_id = connection.id
        break

print(f"The MCP Server Connection ID is: {connection_id}")

The MCP Server Connection ID is: /subscriptions/0bf1e080-6f8c-4569-9e8e-2a1fc5f1caaa/resourceGroups/foundrydemorgzane/providers/Microsoft.CognitiveServices/accounts/udemydemoprojectzane/projects/udemy-demoproj-zane/connections/mslearn-mcp-server


### Creating the MCP Tool Spec

In [6]:
tool = MCPTool(
    server_label = "microsoft_learn_server",
    server_url="https://learn.microsoft.com/api/mcp",
    require_approval="never",
    project_connection_id=connection_id
)

### Creating the MCP Agent

In [7]:
agent = project_client.agents.create_version(
    agent_name="MCP-Agent",
    definition=PromptAgentDefinition(
        model=model_deployment_name,
        instructions="You are an intelligent assistant that can interact with the Microsoft Learn MCP server to provide users with relevant learning resources and information about Microsoft technologies.",
        tools=[tool],
    )
)

print(f"Created MCP Agent with ID: {agent.id}")

Created MCP Agent with ID: MCP-Agent:1


### Creating a Conversation Object for the Agent Chat System

In [8]:
# create a conversation to use with the agent
conversation = openai_client.conversations.create()
print(f"Created conversation with id: {conversation.id}")

Created conversation with id: conv_71914f89b644d3f300qzDriPiZQfLWnhHY87brKrGeCtF7FnYF


### Chat with the Agent

In [9]:
user_query = "Can you pls help me with the latest ai foundry SDK code samples?" 

# Can also try this query: "Find me learning paths on Azure AI services for building intelligent applications."

In [10]:
response = openai_client.responses.create(
    conversation=conversation.id,
    extra_body={
        "agent": {
            "name": "MCP-Agent",
            "type": "agent_reference"
        }
    },
    input=user_query
)

print(f"Agent Response: {response.output_text}")

Agent Response: Here are some recent C# code samples and resources for working with the AI Foundry SDK:

---

### 1. **Using OpenAI SDK with Foundry Local**
This sample demonstrates how to:
- Initialize and configure the Foundry Local Manager.
- Download and register execution providers.
- Retrieve, download, and load a model.
- Start a local web service and interact with it using the OpenAI SDK for chat completions.

#### Code Sample:
```csharp
using Microsoft.AI.Foundry.Local;
using OpenAI;
using System.ClientModel;

var config = new Configuration
{
    AppName = "foundry_local_samples",
    LogLevel = Microsoft.AI.Foundry.Local.LogLevel.Information,
    Web = new Configuration.WebService
    {
        Urls = "http://127.0.0.1:52495"
    }
};

// Initialize the singleton instance.
await FoundryLocalManager.CreateAsync(config, Utils.GetAppLogger());
var mgr = FoundryLocalManager.Instance;

// Ensure Execution Provider (EP) downloads are completed.
await mgr.DownloadAndRegisterEpsAsync